[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Train and Test


## What you will be able to do

Split a table so that the test set is a fair sample of it, see how much a score moves when the split
changes, keep a rare answer in both parts, and save the test set for one last look.

## The idea

### The problem

The **Your First Model** notebook split the stations once, with `random_state=0`, and the linear
model scored 0.888 on the 300 it held back. That number came from one particular choice of 300
stations. Another seed picks other stations, and some sets of 300 are easier to predict than others,
so the same model on the same table can score noticeably higher or lower without anything about the
model changing.

The split can also go wrong in ways that raise no error. A table exported in some order, by
latitude or by the date each station joined the network, keeps that order unless the rows are
shuffled, and taking the last quarter of an ordered table tests the model on one corner of the
country. A rare answer, such as a frost in June at one station in twenty, can end up mostly on one
side of a random split, leaving the test set with too few examples to judge the model on them.

And a test set is easy to wear out. Every time a score on it guides a choice, such as how deep a
tree should be, the test set stops being unseen, and its score drifts upward from what new stations
will show.

### What a split is

> **`train_test_split`** divides its arguments into a training part and a test part, row by row and
> in step. **`test_size`** sets the size of the test part, as a fraction or a count. **`shuffle`**,
> on by default, mixes the rows first, and **`random_state`** fixes how they are mixed, so the same
> seed gives the same split on every run. **`stratify`** takes a column of classes and keeps each
> class's share the same in both parts.

### Why it works that way

A random split makes the test set a sample of the whole table, so its score estimates what the model
would do on any new station drawn the same way. The estimate has a spread, as any sample does, and a
seed makes it repeatable without making it any more accurate. The **Cross-Validation** notebook
narrows the spread by testing on every station once.

Stratifying matters when a class is rare. With 59 frost stations among 1,200, a quarter for testing
should hold about 15, and chance alone can put far fewer there. `stratify` fixes the count, so the
test set has the same share of frost as the whole network.

Keeping a test set for the end means choosing settings on something else. The usual answer is a
third part, a **validation set**, split from the training rows, which is used for comparisons and
choices, while the test set is looked at once when the choices are made.

### Where this shows up

Every evaluation in the rest of this guide starts with a split. The **Preprocessing** notebook
splits before it scales, the **Imbalanced Classes** notebook stratifies every split of June frost,
and the **Hyperparameter Search** notebook automates the choice that a validation set makes here.

### What this notebook covers

- `test_size` as a fraction and as a count
- what `random_state` fixes, and how far the score moves with it
- shuffling a table that arrives in order
- `stratify` for the rare June frost
- a validation set to choose a tree's depth, and the test set used once
- three errors, of which only the first raises an exception

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
for seed in [0, 1, 2]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)
    model = LinearRegression().fit(X_train, y_train)
    print(seed, round(model.score(X_test, y_test), 3))
```

```
0 0.888
1 0.875
2 0.876
```

The same model on the same table scored 0.888, 0.875 and 0.876, because each seed put different
stations in the test set. None of the three is the true score. Together they show how much one split
can be trusted.

## Setup

Two libraries for the data, scikit-learn, and three of its tools.

- `numpy` and `pandas` build the stations, and the cell prints scikit-learn's version
- `train_test_split` divides the stations into parts
- `LinearRegression` and `DecisionTreeRegressor` are the models whose scores the splits change

`station_network` makes the same network of 1,200 weather stations in Norway as **The Estimator
API** notebook. The stations are made up, and a generator started from a fixed seed draws them, so
the table is the same on every run. `FEATURES` names the five numeric columns the models use. `X`
and `y` are the features and the January mean, made here because every section splits them.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
X = network[FEATURES]
y = network["jan_mean"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### The size of the test set

`test_size` takes a fraction of the rows or a number of rows:

In [2]:
for size in [0.25, 0.1, 60]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=size, random_state=0)
    print(f"test_size={size}: {len(X_train)} for training, {len(X_test)} for testing")

test_size=0.25: 900 for training, 300 for testing
test_size=0.1: 1080 for training, 120 for testing
test_size=60: 1140 for training, 60 for testing


A fraction is a share of the 1,200 stations, and a whole number is a count. A larger test set gives
a steadier score and leaves fewer stations to learn from. A quarter is a common choice for a table
this size.

### What the seed fixes

The same seed picks the same stations, and another seed picks others:

In [3]:
first = train_test_split(X, y, test_size=0.25, random_state=0)[1]
again = train_test_split(X, y, test_size=0.25, random_state=0)[1]
other = train_test_split(X, y, test_size=0.25, random_state=1)[1]

print("seed 0, twice, same stations:", first.index.equals(again.index))
print("seed 0 starts with", first.index[:3].tolist(), "and seed 1 with", other.index[:3].tolist())
print("stations in both test sets:", len(first.index.intersection(other.index)))

seed 0, twice, same stations: True
seed 0 starts with [198, 328, 45] and seed 1 with [636, 683, 1033]
stations in both test sets: 77


`[1]` takes the second of the four pieces, the test part of `X`. Two test sets of 300 drawn with
different seeds share about a quarter of their stations, as two random quarters would.

### How far the score moves

In [4]:
scores = []
for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)
    scores.append(LinearRegression().fit(X_train, y_train).score(X_test, y_test))

print([round(score, 3) for score in scores])
print("lowest", round(min(scores), 3), "| highest", round(max(scores), 3))

[0.888, 0.875, 0.876, 0.873, 0.876, 0.901, 0.884, 0.877, 0.856, 0.885]
lowest 0.856 | highest 0.901


Ten splits gave scores from 0.856 to 0.901, so one split pins the model's score on new stations down
only to within a few hundredths. Telling two models apart takes their scores on several splits,
which is the reason the **Cross-Validation** notebook uses several at once.

### Shuffling a table that arrives in order

A register exported station by station from south to north holds its stations in order of latitude.
`train_test_split` shuffles by default, so a random split of the ordered table still reaches every
part of the country:

In [5]:
by_latitude = network.sort_values("latitude")
X_train, X_test, y_train, y_test = train_test_split(by_latitude[FEATURES], by_latitude["jan_mean"],
                                                    test_size=0.25, random_state=0)

linear = LinearRegression().fit(X_train, y_train)
tree = DecisionTreeRegressor(max_depth=6, random_state=0).fit(X_train, y_train)

print("test latitudes from", X_test["latitude"].min(), "to", X_test["latitude"].max())
print("linear R²:", round(linear.score(X_test, y_test), 3))
print("tree R²:  ", round(tree.score(X_test, y_test), 3))
linear_errors = linear.predict(X_test) - y_test
tree_errors = tree.predict(X_test) - y_test
print("mean error, ignoring its sign: linear", round(linear_errors.abs().mean(), 2),
      "| tree", round(tree_errors.abs().mean(), 2))
print("spread of the measured values:", round(y_test.std(), 2))

test latitudes from 58.03 to 70.98
linear R²: 0.885
tree R²:   0.817
mean error, ignoring its sign: linear 0.75 | tree 0.91
spread of the measured values: 2.79


The test stations run from 58.03 to 70.98 degrees north, the whole length of the network. The linear
model's score is in the range the ten seeds gave, a tree limited to six levels scores 0.817, and
their predictions are 0.75 and 0.91 degrees off on average. The Common errors section turns the
shuffle off.

### Keeping the rare answer in both parts

`june_frost` is true at 59 of the 1,200 stations. Given the column, `stratify` splits each class
separately, so both parts keep its share:

In [6]:
frost = network["june_frost"]

for seed in range(3):
    _, _, frost_train, frost_test = train_test_split(X, frost, test_size=0.25, random_state=seed,
                                                     stratify=frost)
    print(f"seed {seed}: {frost_train.sum()} to train on, {frost_test.sum()} to test on")

first_test = train_test_split(X, frost, test_size=0.25, random_state=0, stratify=frost)[1]
second_test = train_test_split(X, frost, test_size=0.25, random_state=1, stratify=frost)[1]
print("stations in both test sets:", len(first_test.index.intersection(second_test.index)))

seed 0: 44 to train on, 15 to test on
seed 1: 44 to train on, 15 to test on
seed 2: 44 to train on, 15 to test on
stations in both test sets: 77


Every seed gives 44 frost stations for training and 15 for testing, a quarter of the 59, and the
test sets of seeds 0 and 1 still share only 77 of their 300 stations. `_` is the usual name for a
piece that is not needed.

### Choosing a depth with a validation set

A tree's depth is a choice, and making it with the test set would spend the test set on it.
Splitting the training rows again gives a validation set to choose with:

In [7]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_fit, X_valid, y_fit, y_valid = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0)

print(len(X_fit), "to fit,", len(X_valid), "to validate,", len(X_test), "kept for the end")

for depth in [2, 4, 6, 8, 10, None]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_fit, y_fit)
    fit_score, valid_score = tree.score(X_fit, y_fit), tree.score(X_valid, y_valid)
    print(f"max_depth={depth}: fit {fit_score:.3f}, validation {valid_score:.3f}")

720 to fit, 240 to validate, 240 kept for the end
max_depth=2: fit 0.578, validation 0.591
max_depth=4: fit 0.819, validation 0.724
max_depth=6: fit 0.917, validation 0.775
max_depth=8: fit 0.972, validation 0.756
max_depth=10: fit 0.992, validation 0.740
max_depth=None: fit 1.000, validation 0.732


The fitting score rises with every level, to 1.0 with no limit, and the validation score peaks at
depth 6, with 0.775, then falls as the tree starts to memorize. So depth 6 is the choice this
validation set makes, with depth 8 close behind at 0.756. The chosen model is then trained on all
960 stations not kept for the end, and the test set is used once:

In [8]:
chosen = DecisionTreeRegressor(max_depth=6, random_state=0).fit(X_rest, y_rest)

print("R² on the test set, looked at once:", round(chosen.score(X_test, y_test), 3))

R² on the test set, looked at once: 0.835


0.835 is the number to report for this tree. The validation scores helped choose, so they were part
of the training, and the test score is the only one no choice was made with.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/03-train-and-test-solutions.ipynb).

**1.** Split `X` and `y` with `test_size=100` and `random_state=0`, and print how many stations went into each part.

In [9]:
# your code here

**2.** For the seeds 10 to 14, train a `LinearRegression` on a quarter-sized split and print each test R², then the lowest and the highest.

In [10]:
# your code here

**3.** Sort `network` by `elevation_m`, split it with `shuffle=False`, and print the elevation range of the test set and the linear model's test R².

In [11]:
# your code here

**4.** Split `june_frost` with `test_size=0.2`, `random_state=0` and `stratify`, and print the number of frost stations in each part.

In [12]:
# your code here

**5.** Choose between `max_depth` 3, 5 and 7 for a `DecisionTreeRegressor(random_state=0)` on a validation set split from the training rows, then train the chosen depth on all the training rows and print its test R² once.

In [13]:
# your code here

## Common errors

### InvalidParameterError: The 'test_size' parameter of train_test_split must be a float in the range (0.0, 1.0), an int in the range [1, inf) or None. Got 1.5 instead.

In [14]:
train_test_split(X, y, test_size=1.5)

InvalidParameterError: The 'test_size' parameter of train_test_split must be a float in the range (0.0, 1.0), an int in the range [1, inf) or None. Got 1.5 instead.

A fraction has to lie between 0 and 1, and 1.5 is neither a fraction nor a whole count of rows. The
message lists what is allowed. `test_size=0.15` keeps 15 percent, and `test_size=150` keeps 150
stations.

### No error, and a test set of only the northern stations

In [15]:
X_train, X_test, y_train, y_test = train_test_split(by_latitude[FEATURES], by_latitude["jan_mean"],
                                                    test_size=0.25, shuffle=False)

print("training latitudes from", X_train["latitude"].min(), "to", X_train["latitude"].max())
print("test latitudes from", X_test["latitude"].min(), "to", X_test["latitude"].max())

linear = LinearRegression().fit(X_train, y_train)
tree = DecisionTreeRegressor(max_depth=6, random_state=0).fit(X_train, y_train)
print("linear R²:", round(linear.score(X_test, y_test), 3))
print("tree R²:  ", round(tree.score(X_test, y_test), 3))

linear_errors = linear.predict(X_test) - y_test
tree_errors = tree.predict(X_test) - y_test
print("mean error, ignoring its sign: linear", round(linear_errors.abs().mean(), 2),
      "| tree", round(tree_errors.abs().mean(), 2))
print("tree's mean error, with its sign:", round(tree_errors.mean(), 2))
print("spread of the measured values:", round(y_test.std(), 2))

latitude_questions = tree.tree_.threshold[tree.tree_.feature == 0]
print("tree's highest latitude question:", round(latitude_questions.max(), 2))

training latitudes from 58.01 to 67.64
test latitudes from 67.65 to 70.98
linear R²: 0.763
tree R²:   0.304
mean error, ignoring its sign: linear 0.83 | tree 1.51
tree's mean error, with its sign: 1.14
spread of the measured values: 2.2
tree's highest latitude question: 66.93


With `shuffle=False`, the split takes the last quarter of the rows as they stand, and in a table
sorted by latitude that is every station from 67.65 degrees north. The models trained only on the
south and were tested only on the north. `tree.tree_` holds the tree's questions, and column 0 of
`FEATURES` is latitude, so the last line shows that its highest question about latitude is at 66.93
degrees. It answers for a station at 70 degrees as it would for the northernmost stations it learned
from, and its predictions come out 1.14 degrees too warm on average. The linear model carries its
latitude coefficient on past the training rows, and its mean error grew only from 0.75 to 0.83
degrees, where the tree's grew from 0.91 to 1.51. The R² scores, 0.763 and 0.304 against 0.885 and
0.817 shuffled, fall further than those errors suggest, because the northern stations' January means
spread less, with a standard deviation of 2.2 degrees against 2.79, and R² measures errors against
that spread. Leave `shuffle` on unless the order is the point, as when testing a forecast on the
most recent days.

### No error, and test sets whose number of frost stations depends on the seed

In [16]:
counts = []
for seed in range(10):
    _, _, _, frost_test = train_test_split(X, frost, test_size=0.25, random_state=seed)
    counts.append(int(frost_test.sum()))

print("frost stations in a test set of 300:", counts)

frost stations in a test set of 300: [21, 12, 14, 18, 10, 19, 15, 14, 20, 16]


Without `stratify`, the 300 test stations held between 10 and 21 frost stations over these ten
seeds, where a quarter of the 59 is about 15. A test set with 10 has too few examples to judge how a
model finds frost, and comparing models across seeds compares different test sets. Pass the class
column as `stratify` whenever a class is rare, as the worked example did.

## Recap

- Choose the size of the test set with `test_size`, as a fraction or a count
- Fix a split with `random_state`, and expect the score to move when the seed does
- Keep `shuffle` on, so that a table in order still gives a test set from every part of it
- Keep a rare class's share in both parts with `stratify`
- Make choices on a validation set, and look at the test set once

## What is next

The **Preprocessing** notebook scales numeric columns and encodes text ones, learning how from the
training rows alone.

---

&#8592; **Previous:** [Your First Model](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/02-your-first-model.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Preprocessing](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/04-preprocessing.ipynb) &#8594;
